# Hybrid Search — Dense + BM25

`04-Evaluate_Formulary_RAG.ipynb` measured pure vector (FAISS + local embeddings) retrieval at 93.5% Hit Rate / 0.854 MRR at `top_k=3`. This notebook follows the same idea as `ai-tutor-rag-system/notebooks/11-Adding_Hybrid_Search.ipynb` — combine dense (semantic) retrieval with a lexical/keyword retriever, since embeddings alone can rank a *semantically similar* row above the row that *exactly matches* the query's tokens — and checks whether it actually improves on this project's own numbers, using the exact same saved eval set from `04` (`eval_set_200.json`).

**Why this is a better fit here than most retrieval upgrades:** a drug name + strength (`"acetaminophen w/ codeine tab 300-15 mg"`) is a precise lexical string, not prose meaning. That's exactly the case dense embeddings sometimes blur across similar drugs in the same class/section, and exactly what exact term matching is good at.

**The one adaptation this requires:** `11-Adding_Hybrid_Search.ipynb` pairs LlamaIndex's vector index with its `SimpleKeywordTableIndex`. This project is LangChain + FAISS, so the lexical half here is LangChain's `BM25Retriever` (statistical term matching via `rank_bm25` — no LLM, no cloud call) combined through `EnsembleRetriever`. Same idea, native to this stack.

**What this does *not* fix:** a user asking about a drug by **brand name** when the formulary lists the **generic name** (or vice versa) won't be helped by either retriever here — that's a vocabulary gap, not a ranking gap, and the right fix is a brand↔generic synonym table, not hybrid search.

## 1. Setup

In [1]:
import yaml
from pathlib import Path

config_path = Path("../../config/config.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print(f"✅ Config loaded. Configured top_k: {config['retrieval']['top_k']}")

✅ Config loaded. Configured top_k: 3


## 2. Load the Cached Vector Store (built by `02-Basic_RAG_Formulary.ipynb`)

Same reuse principle as `04`: no PDF re-parse, no re-embedding. Every row for both retrievers comes straight out of the existing FAISS cache's docstore.

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name=config["embedding"]["model_name"])

MODEL_TAG = config["embedding"]["model_name"].replace("/", "-")
CACHE_DIR = Path(f"../../{config['paths']['processed_data']}/faiss_cache_{MODEL_TAG}_{config['embedding']['dimension']}d")
INDEX_NAME = "formulary_rows"

assert (CACHE_DIR / f"{INDEX_NAME}.faiss").exists(), (
    f"No cache at {CACHE_DIR} — run 02-Basic_RAG_Formulary.ipynb first to build it."
)

vectorstore = FAISS.load_local(
    str(CACHE_DIR), embeddings, index_name=INDEX_NAME, allow_dangerous_deserialization=True
)

all_docs = list(vectorstore.docstore._dict.values())
print(f"✅ Loaded {len(all_docs)} indexed rows from {CACHE_DIR}")

/tmp/ipykernel_28232/520052283.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/home/bhargav/portfolio-project/healthins-enrollment-assistant-dev/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3582.85it/s]

✅ Loaded 2394 indexed rows from ../../data/processed/NC/2026/formulary/faiss_cache_sentence-transformers-all-MiniLM-L6-v2_384d


## 3. Build the BM25 (Lexical) Retriever

Built from the exact same rows as the vector store — statistical term matching, no embeddings, no LLM call.

In [3]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(all_docs)
print("✅ BM25 retriever built over the same indexed rows")

✅ BM25 retriever built over the same indexed rows


## 4. Hybrid Retriever (Ensemble)

`EnsembleRetriever` fuses the two ranked lists via Reciprocal Rank Fusion. Weights are even (0.5/0.5) as a starting point — worth revisiting once real query logs exist, but there's no eval-set evidence yet to justify weighting one side more.

In [4]:
from langchain_classic.retrievers import EnsembleRetriever


def build_hybrid_retriever(k, vector_weight=0.5, bm25_weight=0.5):
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    bm25_retriever.k = k
    return EnsembleRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        weights=[vector_weight, bm25_weight],
    )


# Quick sanity check on a query where the exact drug string should matter a lot.
sample_query = f"What tier and requirements/limits apply to {all_docs[0].metadata['drug']}?"
hybrid = build_hybrid_retriever(k=3)
for doc in hybrid.invoke(sample_query):
    print(doc.metadata["drug"], "| p.", doc.metadata["page"])

ed-spaz tbdp .125mg | p. 73
ed at the p | p. 7
CITRANATAL MIS 90 DHA | p. 91
raloxifene hcl tabs 60mg | p. 72
PRED-G S.O.P OIN OP | p. 95
letrozole tabs 2.5mg | p. 34


## 5. Eval Set (loaded from `04-Evaluate_Formulary_RAG.ipynb`)

Loading the exact saved `eval_set_200.json` — not regenerating it from a duplicated random-sampling snippet — so this is a genuine apples-to-apples comparison against `04`'s numbers rather than two notebooks that merely *intend* to build the same set.

In [5]:
import json

EVAL_SET_PATH = Path(f"../../{config['paths']['processed_data']}/eval_set_200.json")
assert EVAL_SET_PATH.exists(), (
    f"No eval set at {EVAL_SET_PATH} — run 04-Evaluate_Formulary_RAG.ipynb first to build it."
)
with open(EVAL_SET_PATH) as f:
    eval_set = json.load(f)

print(f"✅ Loaded {len(eval_set)} eval questions from {EVAL_SET_PATH}")

✅ Loaded 200 eval questions from ../../data/processed/NC/2026/formulary/eval_set_200.json


## 6. Retrieval Evaluation: Vector-Only vs. Hybrid, Hit Rate & MRR vs. `top_k`

In [6]:
import pandas as pd


def _rank_of_expected(docs, item, k):
    for i, doc in enumerate(docs[:k], start=1):
        if (
            doc.metadata.get("drug") == item["expected_drug"]
            and doc.metadata.get("page") == item["expected_page"]
        ):
            return i
    return None


def evaluate_vector(eval_set, k):
    hits, rrs = [], []
    for item in eval_set:
        docs = vectorstore.similarity_search(item["question"], k=k)
        rank = _rank_of_expected(docs, item, k)
        hits.append(1 if rank else 0)
        rrs.append(1 / rank if rank else 0)
    return {"top_k": k, "hit_rate": sum(hits) / len(hits), "mrr": sum(rrs) / len(rrs)}


def evaluate_hybrid(eval_set, k):
    retriever = build_hybrid_retriever(k)
    hits, rrs = [], []
    for item in eval_set:
        docs = retriever.invoke(item["question"])
        rank = _rank_of_expected(docs, item, k)
        hits.append(1 if rank else 0)
        rrs.append(1 / rank if rank else 0)
    return {"top_k": k, "hit_rate": sum(hits) / len(hits), "mrr": sum(rrs) / len(rrs)}


rows = []
for k in [1, 2, 3, 5, 10]:
    v = evaluate_vector(eval_set, k)
    h = evaluate_hybrid(eval_set, k)
    rows.append({"top_k": k, "retriever": "vector-only", **{kk: vv for kk, vv in v.items() if kk != "top_k"}})
    rows.append({"top_k": k, "retriever": "hybrid", **{kk: vv for kk, vv in h.items() if kk != "top_k"}})

results_df = pd.DataFrame(rows)[["top_k", "retriever", "hit_rate", "mrr"]]
results_df

,top_k,retriever,hit_rate,mrr
0,1,vector-only,0.785,0.785000
1,1,hybrid,0.785,0.785000
2,2,vector-only,0.900,0.842500
3,2,hybrid,0.935,0.895000
4,3,vector-only,0.935,0.854167
5,3,hybrid,0.980,0.922500
6,5,vector-only,0.980,0.864417
7,5,hybrid,1.000,0.942667
8,10,vector-only,0.980,0.864417
9,10,hybrid,1.000,0.937667


In [7]:
pivot = results_df.pivot(index="top_k", columns="retriever", values=["hit_rate", "mrr"])
pivot

hit_rate                   mrr            
retriever   hybrid vector-only    hybrid vector-only
top_k                                               
1            0.785       0.785  0.785000    0.785000
2            0.935       0.900  0.895000    0.842500
3            0.980       0.935  0.922500    0.854167
5            1.000       0.980  0.942667    0.864417
10           1.000       0.980  0.937667    0.864417

## 🔑 Key Takeaways

- **Compare on the identical eval set, not a fresh sample.** Reusing `04`'s exact seed and question-template means any difference in the table above is attributable to the retriever, not to eval-set luck.
- **Hybrid search here is free in every sense that matters.** `BM25Retriever` runs on CPU with no model download and no LLM call — the only cost is a small amount of extra retrieval latency, not infrastructure or API spend.
- **This targets a specific failure mode, not general retrieval quality.** If hybrid's gain is concentrated in cases where the query contains the drug's exact name/strength tokens, that confirms the mechanism (lexical matching rescuing exact-string queries) rather than hybrid search being a blanket upgrade.
- **Brand-name mismatches are still out of scope.** Neither retriever here helps if the query uses a brand name absent from the formulary text — that gap needs a synonym table, a separate and still-unbuilt piece of work.
- **Decide from the numbers above, not from this notebook's premise.** If hybrid doesn't beat vector-only at your chosen `top_k` on this data, the honest conclusion is to keep the simpler vector-only pipeline — added complexity needs to earn its place.